In [ ]:
import requests
import pandas as pd

def fetch_historical_rainfall(lat=5.3, lon=103.1, start='2023-01-01', end='2024-12-31'):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={start}&end_date={end}"
        f"&daily=precipitation_sum&timezone=Asia/Kuala_Lumpur"
    )
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json().get('daily', {})
    return pd.DataFrame({
        'date': data['time'],
        'rainfall_mm': data['precipitation_sum']
    })

# Example usage
df_weather = fetch_historical_rainfall(start='2020-01-01', end='2025-08-01')
print(df_weather.head())


         date  rainfall_mm
0  2020-01-01          4.2
1  2020-01-02          8.7
2  2020-01-03          7.5
3  2020-01-04          2.8
4  2020-01-05          2.8


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Growth stages with scheduling for both methods (in Days After Seeding/Transplanting)
GROWTH_STAGES = {
    "Transplanting": [
        ("Transplanting", 0),
        ("Germination", 1),
        ("Early Vegetative", 5),
        ("Vegetative Growth", 8),
        ("Tillering", 25),
        ("Panicle Initiation", 45),
        ("Flowering", 65),
        ("Grain Filling", 71),
        ("Ripening", 101),
        ("Harvest", 110)
    ],
    "Direct Seeding": [
        ("Germination", 0),
        ("Emergence", 4),
        ("Pre-AWD Transition", 11),
        ("Early Vegetative", 15),
        ("Tillering", 25),
        ("Panicle Initiation", 45),
        ("Flowering", 65),
        ("Grain Filling", 71),
        ("Ripening", 101),
        ("Harvest", 110)
    ]
}

# Stage-specific ponding depths (upper water levels in cm)
WATER_THRESHOLDS = {
    "Transplanting": {
        "Transplanting": 5,
        "Flowering": 5,
    },
    "Direct Seeding": {
        "Germination": 5,
        "Emergence": 3,
        "Flowering": 5,
    }
}

def get_stage(method, day):
    stages = GROWTH_STAGES[method]
    for i in range(len(stages)-1, -1, -1):
        if day >= stages[i][1]:
            return stages[i][0]
    return "Unknown"

def get_irrigation_label(method, phase, stage, rainfall, current_water_level):
    # If stage has a specific threshold, use it; otherwise use phase-based default
    if stage in WATER_THRESHOLDS.get(method, {}):
        threshold = WATER_THRESHOLDS[method][stage]
    else:
        threshold = 5 if phase == "Flooding" else -15

    # No irrigation during these stages
    if stage in ['Ripening', 'Harvest']:
        return 0

    # Convert rainfall from mm to cm for comparison
    rainfall_cm = rainfall / 10.0
    diff = abs(threshold - current_water_level)

    # Irrigation decision
    if current_water_level < threshold and rainfall_cm < diff:
        return 1
    else:
        return 0


def generate_data_from_weather(days):
    records = []
    methods = ["Transplanting", "Direct Seeding"]

    for method in methods:
        for idx in range(days):
            days = random.randint(0, 110)  # random day in crop cycle
            stage = get_stage(method, days)

            rainfall = df_weather.loc[idx, 'rainfall_mm']
            current_water_level = round(np.random.uniform(-13, 7), 1)

            if stage in ('Transplanting', 'Flowering', 'Germination', 'Emergence'):
                phase = 'Flooding'
            elif stage in ('Ripening', 'Harvest'):
                phase = 'Drying'
            else:
                phase = random.choice(["Drying", "Flooding"])

            irrigation = get_irrigation_label(method, phase, stage, rainfall, current_water_level)

            records.append({
                "Date": df_weather.loc[idx, 'date'],  # use weather date
                "Method": method,
                "Days": days,
                "GrowthStage": stage,
                "Rainfall_mm": rainfall,
                "CurrentWaterLevel_cm": current_water_level,
                "IrrigationPhase": phase,
                "IrrigationNeeded": irrigation
            })

    return pd.DataFrame(records)


if __name__ == "__main__":
    days = len(df_weather)
    df = generate_data_from_weather(days)
    df.to_csv("simulated_rice_irrigation_data.csv", index=False)
    print("Dataset saved as simulated_rice_irrigation_data.csv")
    print(df.head(10))

    irrigation_on = (df['IrrigationNeeded'] == 1).sum()
    irrigation_off = (df['IrrigationNeeded'] == 0).sum()
    print(f"Irrigation ON (1): {irrigation_on}")
    print(f"Irrigation OFF (0): {irrigation_off}")


Dataset saved as simulated_rice_irrigation_data.csv
         Date         Method  Days         GrowthStage  Rainfall_mm  \
0  2020-01-01  Transplanting    81       Grain Filling          4.2   
1  2020-01-02  Transplanting    87       Grain Filling          8.7   
2  2020-01-03  Transplanting    47  Panicle Initiation          7.5   
3  2020-01-04  Transplanting    69           Flowering          2.8   
4  2020-01-05  Transplanting    14   Vegetative Growth          2.8   
5  2020-01-06  Transplanting    62  Panicle Initiation          1.2   
6  2020-01-07  Transplanting    63  Panicle Initiation          4.8   
7  2020-01-08  Transplanting    82       Grain Filling          4.5   
8  2020-01-09  Transplanting   105            Ripening          3.3   
9  2020-01-10  Transplanting    58  Panicle Initiation          3.2   

   CurrentWaterLevel_cm IrrigationPhase  IrrigationNeeded  
0                  -4.1          Drying                 0  
1                   3.1        Flooding       

In [ ]:

import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("simulated_rice_irrigation_data.csv")

# Encode categorical features
le_method = LabelEncoder()
df['Method'] = le_method.fit_transform(df['Method'])

le_phase = LabelEncoder()
df['IrrigationPhase'] = le_phase.fit_transform(df['IrrigationPhase'])

# Encode GrowthStage as categorical integers
le_stage = LabelEncoder()
df['GrowthStage'] = le_stage.fit_transform(df['GrowthStage'])

# Features and target
X = df[['Method', 'Days', 'GrowthStage', 'Rainfall_mm', 'CurrentWaterLevel_cm', 'IrrigationPhase']]
y = df['IrrigationNeeded']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost model
model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Threshold mapping reused during inference
try:
    WATER_THRESHOLDS
except NameError:
    WATER_THRESHOLDS = {
        "Transplanting": {
            "Transplanting": 5,
            "Flowering": 5,
        },
        "Direct Seeding": {
            "Germination": 5,
            "Emergence": 3,
            "Flowering": 5,
        },
    }

class IrrigationModelWithThreshold:
    """Wraps the classifier so predict returns [decision, threshold]."""

    def __init__(self, model, le_method, le_phase, le_stage, thresholds, default_flood=5, default_dry=-15):
        self.model = model
        self.le_method = le_method
        self.le_phase = le_phase
        self.le_stage = le_stage
        self.thresholds = thresholds
        self.default_flood = default_flood
        self.default_dry = default_dry

    def _compute_threshold(self, method, stage, phase):
        if method in self.thresholds and stage in self.thresholds[method]:
            return self.thresholds[method][stage]
        return self.default_flood if phase == "Flooding" else self.default_dry

    def predict(self, X):
        # Accept DataFrame or array-like with the same column order used in training.
        if isinstance(X, pd.DataFrame):
            arr = X[['Method', 'Days', 'GrowthStage', 'Rainfall_mm', 'CurrentWaterLevel_cm', 'IrrigationPhase']].to_numpy()
        else:
            arr = np.asarray(X)

        decisions = self.model.predict(arr)

        method_codes = arr[:, 0].astype(int)
        stage_codes = arr[:, 2].astype(int)
        phase_codes = arr[:, 5].astype(int)

        methods = self.le_method.inverse_transform(method_codes)
        stages = self.le_stage.inverse_transform(stage_codes)
        phases = self.le_phase.inverse_transform(phase_codes)

        outputs = []
        for dec, meth, stg, ph in zip(decisions, methods, stages, phases):
            thr = self._compute_threshold(meth, stg, ph)
            outputs.append([int(dec), float(thr)])
        return np.array(outputs)

# Save wrapped model + encoders
wrapped_model = IrrigationModelWithThreshold(model, le_method, le_phase, le_stage, WATER_THRESHOLDS)
joblib.dump(wrapped_model, "irrigation_xgb_model.pkl")
joblib.dump((le_method, le_phase, le_stage), "label_encoders.pkl")


Accuracy: 0.9976580796252927
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       243
           1       1.00      0.99      1.00       184

    accuracy                           1.00       427
   macro avg       1.00      1.00      1.00       427
weighted avg       1.00      1.00      1.00       427



['label_encoders.pkl']

In [ ]:

import requests
from datetime import datetime
import joblib
import pandas as pd

# Load model and encoders
model = joblib.load("irrigation_xgb_model.pkl")
le_method, le_phase, le_stage = joblib.load("label_encoders.pkl")

# Growth stages dictionary (same as before)
GROWTH_STAGES = {
    "Transplanting": [
        ("Transplanting", 0),
        ("Germination", 1),
        ("Early Vegetative", 5),
        ("Vegetative Growth", 8),
        ("Tillering", 25),
        ("Panicle Initiation", 45),
        ("Flowering", 65),
        ("Grain Filling", 71),
        ("Ripening", 101),
        ("Harvest", 110)
    ],
    "Direct Seeding": [
        ("Germination", 0),
        ("Emergence", 4),
        ("Pre-AWD Transition", 11),
        ("Early Vegetative", 15),
        ("Tillering", 25),
        ("Panicle Initiation", 45),
        ("Flowering", 65),
        ("Grain Filling", 71),
        ("Ripening", 101),
        ("Harvest", 110)
    ]
}  # same as in your dataset generator

# Stage-specific ponding thresholds (cm)
WATER_THRESHOLDS = {
    "Transplanting": {
        "Transplanting": 5,
        "Flowering": 5,
    },
    "Direct Seeding": {
        "Germination": 5,
        "Emergence": 3,
        "Flowering": 5,
    },
}

def get_stage(method, day):
    stages = GROWTH_STAGES[method]
    for i in range(len(stages) - 1, -1, -1):
        if day >= stages[i][1]:
            return stages[i][0]
    return "Unknown"

def get_threshold(method, stage, phase):
    stage_thresholds = WATER_THRESHOLDS.get(method, {})
    if stage in stage_thresholds:
        return stage_thresholds[stage]
    return 5 if phase == "Flooding" else -15

def fetch_forecast(lat=5.3, lon=103.1):
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&daily=precipitation_sum"
        f"&timezone=Asia/Kuala_Lumpur"
    )
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json()
    return data['daily']['precipitation_sum'][0]  # today's forecast in mm

def predict_irrigation(method, start_date, current_water_level, phase, lat=5.3, lon=103.1):
    today = datetime.now().date()
    days = (today - start_date).days
    stage = get_stage(method, days)
    rainfall_mm = fetch_forecast(lat, lon)
    threshold = get_threshold(method, stage, phase)

    # Encode inputs
    method_enc = le_method.transform([method])[0]
    phase_enc = le_phase.transform([phase])[0]
    stage_enc = le_stage.transform([stage])[0]

    X_input = pd.DataFrame([
        [
            method_enc,
            days,
            stage_enc,
            rainfall_mm,
            current_water_level,
            phase_enc,
        ]
    ], columns=['Method', 'Days', 'GrowthStage', 'Rainfall_mm', 'CurrentWaterLevel_cm', 'IrrigationPhase'])

    prediction_with_threshold = model.predict(X_input)[0]
    decision, model_threshold = prediction_with_threshold
    return decision, model_threshold

# Example
start_date = datetime(2024, 6, 1).date()
decision, threshold = predict_irrigation("Transplanting", start_date, current_water_level=5, phase="Drying")
print("Irrigation needed?", "YES" if decision == 1 else "NO", "| Threshold:", f"{threshold} cm")


Irrigation needed? NO | Threshold: -15.0 cm
